# E. coli estimation error analysis

This notebook analyzes raw per-directed-edge outputs from run `20260805_174708`. It reports descriptive edge and node aggregate error, deterministic high-error rankings, parameter responses, and the requested interaction model. Missing or failed estimates are excluded from metrics but retained in quality counts. Node incident sums assign every directed edge to both endpoints; incident means provide a degree-adjusted sensitivity analysis.

In [1]:
import json
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt

REPO = Path.cwd().resolve()
while not (REPO / "src" / "nocap").exists() and REPO != REPO.parent:
    REPO = REPO.parent
RUN_DIR = REPO / "notebooks/Ecoli_Analysis_Notebooks/estimation/20260805_174708"
INPUT_DIR = RUN_DIR / "csv"
OUT_DIR = RUN_DIR / "error"
OUT_DIR.mkdir(parents=True, exist_ok=True)
assert INPUT_DIR.is_dir(), INPUT_DIR
csv_paths = sorted(INPUT_DIR.glob("*.csv"))
assert csv_paths, f"No CSV files in {INPUT_DIR}"
with open(RUN_DIR / "run_metadata.json") as f:
    run_metadata = json.load(f)
try:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf

    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False
print(f"{len(csv_paths)} CSV files; statsmodels={HAS_STATSMODELS}")

336 CSV files; statsmodels=True


In [2]:
REQUIRED = {
    "cause",
    "effect",
    "status",
    "run_status",
    "scm_seed",
    "data_seed",
    "parameter_condition_id",
    "target_effect_id",
    "n_samples",
    "missing_edge_rate",
    "missing_data_rate",
    "missing_data_mechanism",
    "estimated_path_coefficient",
    "ground_truth_beta",
    "error",
}
frames, schema_rows = [], []
for path in csv_paths:
    try:
        frame = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        print(f"File {path} is empty.")
        continue
    missing = sorted(REQUIRED - set(frame.columns))
    schema_rows.append(
        {"file": path.name, "rows": len(frame), "missing_columns": ",".join(missing)}
    )
    assert not missing, f"{path.name}: missing {missing}"
    frame["_source_file"] = path.name
    frames.append(frame)
raw = pd.concat(frames, ignore_index=True)
numeric = [
    "n_samples",
    "missing_edge_rate",
    "missing_data_rate",
    "scm_seed",
    "data_seed",
    "estimated_path_coefficient",
    "ground_truth_beta",
]
for col in numeric:
    raw[col] = pd.to_numeric(raw[col], errors="coerce")
raw["edge"] = raw["cause"].astype(str) + "->" + raw["effect"].astype(str)
raw["log_n"] = np.log(raw["n_samples"])
raw["coefficient_error"] = raw["estimated_path_coefficient"] - raw["ground_truth_beta"]
raw["abs_error"] = raw["coefficient_error"].abs()
raw["sq_error"] = raw["coefficient_error"] ** 2
usable = (
    raw["run_status"].eq("complete")
    & raw["status"].isin(["identifiable", "insufficient_data"])
    & raw["estimated_path_coefficient"].notna()
    & raw["ground_truth_beta"].notna()
    & np.isfinite(raw["coefficient_error"])
)
quality = (
    raw.assign(usable=usable)
    .groupby(["run_status", "status", "usable"], dropna=False)
    .size()
    .reset_index(name="rows")
)
quality.to_csv(OUT_DIR / "error_analysis_data_quality.csv", index=False)
edge = raw.loc[usable].copy()
print(
    "raw rows:",
    len(raw),
    "usable:",
    len(edge),
    "unique scm seeds:",
    edge.scm_seed.nunique(),
    "data seeds:",
    edge.data_seed.nunique(),
)
display(quality)

File /Users/colin.pannikkat/projects/nocap/notebooks/Ecoli_Analysis_Notebooks/estimation/20260805_174708/csv/csd-20260805_174708_paired_hierarchical_0_0_9fcbe7390717f9dc_observational.csv is empty.
File /Users/colin.pannikkat/projects/nocap/notebooks/Ecoli_Analysis_Notebooks/estimation/20260805_174708/csv/csd-20260805_174708_paired_hierarchical_0_0_a27a5fca1f34e102_observational.csv is empty.
File /Users/colin.pannikkat/projects/nocap/notebooks/Ecoli_Analysis_Notebooks/estimation/20260805_174708/csv/csd-20260805_174708_paired_hierarchical_0_0_a89a61252272c3d0_observational.csv is empty.
File /Users/colin.pannikkat/projects/nocap/notebooks/Ecoli_Analysis_Notebooks/estimation/20260805_174708/csv/csd-20260805_174708_paired_hierarchical_0_0_a8c29722d5e91a3f_observational.csv is empty.
File /Users/colin.pannikkat/projects/nocap/notebooks/Ecoli_Analysis_Notebooks/estimation/20260805_174708/csv/csd-20260805_174708_paired_hierarchical_0_0_ab105ca2a3ff1948_observational.csv is empty.
File /User

,run_status,status,usable,rows
0,complete,estimation_error,False,652
1,complete,identifiable,True,825758


In [3]:
PARAMS = [
    "n_samples",
    "log_n",
    "missing_data_rate",
    "missing_edge_rate",
    "missing_data_mechanism",
    "scm_seed",
    "data_seed",
    "parameter_condition_id",
]
GROUP = [
    "scm_seed",
    "data_seed",
    "parameter_condition_id",
    "n_samples",
    "missing_data_rate",
    "missing_edge_rate",
    "missing_data_mechanism",
]
edge_group = edge.groupby(GROUP + ["edge", "cause", "effect"], dropna=False)
edge_table = edge_group.agg(
    mean_abs_error=("abs_error", "mean"),
    total_abs_error=("abs_error", "sum"),
    rmse=("sq_error", lambda x: np.sqrt(x.mean())),
    signed_bias=("coefficient_error", "mean"),
    n=("abs_error", "size"),
    mean_true_beta=("ground_truth_beta", "mean"),
    mean_estimated_beta=("estimated_path_coefficient", "mean"),
).reset_index()
totals = edge_table.groupby(GROUP, dropna=False)["total_abs_error"].transform("sum")
edge_table["total_error_share"] = edge_table["total_abs_error"] / totals.replace(0, np.nan)
edge_summary = (
    edge_table.groupby(GROUP, dropna=False)
    .agg(
        edge_mean_abs_error=("mean_abs_error", "mean"),
        edge_total_abs_error=("total_abs_error", "sum"),
        edge_rmse=("rmse", lambda x: np.sqrt(np.average(x**2))),
        edge_signed_bias=("signed_bias", "mean"),
        edge_valid_edges=("edge", "nunique"),
        edge_valid_rows=("n", "sum"),
    )
    .reset_index()
)

endpoint = pd.concat(
    [
        edge.assign(node=edge["cause"], role="cause"),
        edge.assign(node=edge["effect"], role="effect"),
    ],
    ignore_index=True,
)
node_group = endpoint.groupby(GROUP + ["node"], dropna=False)
node_table = node_group.agg(
    incident_abs_error_sum=("abs_error", "sum"),
    incident_abs_error_mean=("abs_error", "mean"),
    incident_rmse=("sq_error", lambda x: np.sqrt(x.mean())),
    incident_signed_bias=("coefficient_error", "mean"),
    incident_edge_count=("edge", "size"),
    degree_in=("role", lambda x: (x == "effect").sum()),
    degree_out=("role", lambda x: (x == "cause").sum()),
    roles=("role", lambda x: "+".join(sorted(set(x)))),
).reset_index()
node_totals = node_table.groupby(GROUP, dropna=False)["incident_abs_error_sum"].transform("sum")
node_table["node_error_share"] = node_table["incident_abs_error_sum"] / node_totals.replace(
    0, np.nan
)
node_summary = (
    node_table.groupby(GROUP, dropna=False)
    .agg(
        node_incident_sum=("incident_abs_error_sum", "sum"),
        node_incident_mean=("incident_abs_error_mean", "mean"),
        node_incident_rmse=("incident_rmse", lambda x: np.sqrt(np.average(x**2))),
        node_valid_nodes=("node", "nunique"),
        node_valid_rows=("incident_edge_count", "sum"),
    )
    .reset_index()
)
edge_table.to_csv(OUT_DIR / "edge_error_by_graph_condition.csv", index=False)
node_table.to_csv(OUT_DIR / "node_error_by_graph_condition.csv", index=False)
edge_summary.to_csv(OUT_DIR / "edge_aggregate_summary.csv", index=False)
node_summary.to_csv(OUT_DIR / "node_aggregate_summary.csv", index=False)
assert np.allclose(
    edge_table.groupby(GROUP)["total_abs_error"].sum().values,
    edge_summary["edge_total_abs_error"].values,
)
print(edge_summary.shape, node_summary.shape)

(326, 13) (326, 12)


In [4]:
def ranks(table, value, item, prefix):
    x = (
        table.groupby(item, dropna=False)
        .agg(
            aggregate_error=(value, "sum"),
            mean_error=(value, "mean"),
            prevalence=(value, "size"),
            mean_rank=(value, lambda z: np.nan),
        )
        .reset_index()
    )
    x = x.sort_values(
        ["aggregate_error", item], ascending=[False, True], kind="mergesort"
    ).reset_index(drop=True)
    x["global_rank"] = np.arange(1, len(x) + 1)
    x.to_csv(OUT_DIR / f"{prefix}_global_ranking.csv", index=False)
    return x


edge_rank = ranks(edge_table, "total_abs_error", "edge", "edge")
node_rank = ranks(node_table, "incident_abs_error_sum", "node", "node")
for table, value, item, prefix in [
    (edge_table, "total_abs_error", "edge", "edge"),
    (node_table, "incident_abs_error_sum", "node", "node"),
]:
    cell = table.sort_values(
        [*GROUP, value, item], ascending=[True] * len(GROUP) + [False, True], kind="mergesort"
    ).copy()
    cell["rank"] = cell.groupby(GROUP, dropna=False).cumcount() + 1
    cell["cell_fraction"] = cell["rank"] / cell.groupby(GROUP, dropna=False)[item].transform("size")
    cell.to_csv(OUT_DIR / f"{prefix}_cell_rankings.csv", index=False)
    stability = (
        cell.groupby(item)
        .agg(
            cells=(item, "size"),
            top_1pct=("cell_fraction", lambda x: (x <= 0.01).mean()),
            top_5pct=("cell_fraction", lambda x: (x <= 0.05).mean()),
            top_10pct=("cell_fraction", lambda x: (x <= 0.10).mean()),
            mean_rank=("rank", "mean"),
            median_rank=("rank", "median"),
        )
        .reset_index()
        .sort_values(["top_10pct", "mean_rank", item], ascending=[False, True, True])
    )
    stability.to_csv(OUT_DIR / f"{prefix}_stability.csv", index=False)
display(edge_rank.head(20))
display(node_rank.head(20))

,edge,aggregate_error,mean_error,prevalence,mean_rank,global_rank
0,hupB->tyrP,378.712033,1.161693,326,NaN,1
1,modE->oppA,95.898297,0.294167,326,NaN,2
2,xynR->yagF,94.248281,0.289105,326,NaN,3
3,nanR->nanE,92.870511,0.284879,326,NaN,4
4,btsR->csgD,92.638867,0.284168,326,NaN,5
5,narL->nikA,90.811811,0.278564,326,NaN,6
6,nhaR->rrfF,89.368866,0.274138,326,NaN,7
7,pgrR->ycjX,88.816659,0.272444,326,NaN,8
8,fliA->ispE,88.556537,0.271646,326,NaN,9
9,narL->napA,84.306744,0.258610,326,NaN,10


,node,aggregate_error,mean_error,prevalence,mean_rank,global_rank
0,rpoD,10816.422586,33.179210,326,NaN,1
1,nac,9141.048559,28.040026,326,NaN,2
2,narL,5270.559352,16.167360,326,NaN,3
3,glaR,4876.258116,14.957847,326,NaN,4
4,nsrR,3667.987736,11.251496,326,NaN,5
5,lrp,3545.931068,10.877089,326,NaN,6
6,pdhR,2386.158760,7.319505,326,NaN,7
7,narP,2044.283266,6.270808,326,NaN,8
8,modE,1971.337231,6.047047,326,NaN,9
9,nagC,1890.937991,5.800423,326,NaN,10


In [5]:
summary = edge_summary.merge(node_summary, on=GROUP, how="outer")
summary.to_csv(OUT_DIR / "parameter_response_summary.csv", index=False)
plt.close("all")
for metric, ylabel, filename in [
    ("edge_mean_abs_error", "Edge mean absolute error", "edge_mean_error_by_sample.png"),
    ("edge_total_abs_error", "Edge total absolute error", "edge_total_error_by_sample.png"),
    ("node_incident_mean", "Node incident mean error", "node_mean_error_by_sample.png"),
    ("node_incident_sum", "Node incident sum error", "node_total_error_by_sample.png"),
]:
    fig, ax = plt.subplots(figsize=(10, 6))
    for (mechanism, miss_edge), g in summary.groupby(
        ["missing_data_mechanism", "missing_edge_rate"]
    ):
        g = g.sort_values("n_samples")
        ax.plot(g.n_samples, g[metric], marker="o", label=f"{mechanism}, edge={miss_edge:g}")
    ax.set_xscale("log")
    ax.set_xlabel("n_samples")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=7, ncol=2)
    fig.tight_layout()
    fig.savefig(OUT_DIR / filename, dpi=150)
    plt.close(fig)
for metric, filename in [
    ("edge_mean_abs_error", "edge_error_heatmap.png"),
    ("node_incident_mean", "node_error_heatmap.png"),
]:
    pivot = summary.pivot_table(
        index="missing_data_rate", columns="missing_edge_rate", values=metric, aggfunc="mean"
    )
    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(pivot.values, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)), [f"{x:g}" for x in pivot.columns])
    ax.set_yticks(range(len(pivot.index)), [f"{x:g}" for x in pivot.index])
    ax.set_xlabel("missing_edge_rate")
    ax.set_ylabel("missing_data_rate")
    fig.colorbar(im, ax=ax, label=metric)
    fig.tight_layout()
    fig.savefig(OUT_DIR / filename, dpi=150)
    plt.close(fig)

In [6]:
# Requested model: L ~ log(n) + r_miss + d_edge + log(n):r_miss + r_miss:d_edge.
# Predictors are centered/scaled; mechanism is categorical and retained in the pooled fit.
model_rows = summary.copy()
for col in ["log_n", "missing_data_rate", "missing_edge_rate"]:
    model_rows[col] = (np.log(model_rows.n_samples) if col == "log_n" else model_rows[col]).astype(
        float
    )
    model_rows[col + "_z"] = (model_rows[col] - model_rows[col].mean()) / model_rows[col].std(
        ddof=0
    )
model_rows["mechanism"] = model_rows["missing_data_mechanism"].astype("category")
formula = "L ~ log_n_z + missing_data_rate_z + missing_edge_rate_z + log_n_z:missing_data_rate_z + missing_data_rate_z:missing_edge_rate_z + C(mechanism)"
fits, coefficient_rows, diagnostics = [], [], []
for outcome in [
    "edge_mean_abs_error",
    "edge_total_abs_error",
    "node_incident_mean",
    "node_incident_sum",
]:
    d = model_rows.rename(columns={outcome: "L"}).dropna(subset=["L", "scm_seed"])
    record = {
        "outcome": outcome,
        "formula": formula,
        "n": len(d),
        "n_graphs": d.scm_seed.nunique(),
        "u_graph_estimable": d.scm_seed.nunique() >= 2,
        "status": "not_fit",
    }
    if HAS_STATSMODELS and len(d) >= 10:
        try:
            if record["u_graph_estimable"]:
                fit = smf.mixedlm(formula, d, groups=d["scm_seed"]).fit(
                    reml=False, method="lbfgs", disp=False
                )
                record["model_type"] = "MixedLM"
                record["status"] = "fit"
            else:
                fit = smf.ols(formula, d).fit(cov_type="HC3")
                record["model_type"] = "OLS_HC3_fallback"
                record["status"] = "fit; random effect not identifiable"
            for name, coef, se, p, lo, hi in zip(
                fit.params.index,
                fit.params,
                fit.bse,
                fit.pvalues,
                fit.conf_int()[0],
                fit.conf_int()[1],
            ):
                coefficient_rows.append(
                    {
                        "outcome": outcome,
                        "term": name,
                        "estimate": coef,
                        "std_error": se,
                        "p_value": p,
                        "ci_low": lo,
                        "ci_high": hi,
                        "model_type": record["model_type"],
                    }
                )
            fitted = fit.fittedvalues
            residual = d.L - fitted
            diagnostics.append(
                pd.DataFrame(
                    {
                        "outcome": outcome,
                        "observed": d.L,
                        "fitted": fitted,
                        "residual": residual,
                        "scm_seed": d.scm_seed,
                    }
                )
            )
            record["r_squared"] = getattr(fit, "rsquared", np.nan)
        except Exception as exc:
            record["status"] = "fit_failed: " + str(exc)[:300]
    else:
        record["status"] = "statsmodels unavailable or insufficient rows"
    fits.append(record)
pd.DataFrame(fits).to_csv(OUT_DIR / "interaction_model_summary.csv", index=False)
pd.DataFrame(coefficient_rows).to_csv(OUT_DIR / "interaction_model_coefficients.csv", index=False)
if diagnostics:
    pd.concat(diagnostics, ignore_index=True).to_csv(
        OUT_DIR / "interaction_model_diagnostics.csv", index=False
    )
print(pd.DataFrame(fits).to_string(index=False))

             outcome                                                                                                                                        formula   n  n_graphs  u_graph_estimable                              status       model_type  r_squared
 edge_mean_abs_error L ~ log_n_z + missing_data_rate_z + missing_edge_rate_z + log_n_z:missing_data_rate_z + missing_data_rate_z:missing_edge_rate_z + C(mechanism) 326         1              False fit; random effect not identifiable OLS_HC3_fallback   0.853602
edge_total_abs_error L ~ log_n_z + missing_data_rate_z + missing_edge_rate_z + log_n_z:missing_data_rate_z + missing_data_rate_z:missing_edge_rate_z + C(mechanism) 326         1              False fit; random effect not identifiable OLS_HC3_fallback   0.853602
  node_incident_mean L ~ log_n_z + missing_data_rate_z + missing_edge_rate_z + log_n_z:missing_data_rate_z + missing_data_rate_z:missing_edge_rate_z + C(mechanism) 326         1              False fit; random effect n

## Adjustment-set and coefficient-recovery analysis

The following analysis tests whether recorded adjustment-set size and individual adjustment-node membership are associated with coefficient recovery error. Empty sets are retained, and failed estimates are excluded from error metrics but retained in quality counts. These are descriptive associations, not causal effects, because adjustment-set choice is correlated with graph and edge structure.

In [13]:
import ast


def parse_adjustment_set(value):
    if pd.isna(value):
        return tuple()
    text = str(value).strip()
    if not text or text in {"[]", "{}", "nan", "None"}:
        return tuple()
    try:
        parsed = ast.literal_eval(text)
    except (ValueError, SyntaxError):
        parsed = [x.strip() for x in text.split("|") if x.strip()]
    if isinstance(parsed, str):
        parsed = [parsed]
    return tuple(sorted(map(str, parsed)))


adjustment = raw.copy()
adjustment["adjustment_nodes"] = adjustment["adjustment_set"].map(parse_adjustment_set)
adjustment["adjustment_set_key"] = adjustment["adjustment_nodes"].map(
    lambda x: "|".join(x) if x else "<empty>"
)
adjustment["adjustment_set_size"] = adjustment["adjustment_nodes"].map(lambda x: len(x))
adjustment["coefficient_error"] = (
    adjustment["estimated_path_coefficient"] - adjustment["ground_truth_beta"]
)
adjustment["abs_coefficient_error"] = adjustment["coefficient_error"].abs()
adjustment["usable_error"] = (
    adjustment["run_status"].eq("complete")
    & adjustment["status"].isin(["identifiable", "insufficient_data"])
    & adjustment["coefficient_error"].notna()
    & np.isfinite(adjustment["coefficient_error"])
)

set_quality = (
    adjustment.groupby(["adjustment_set_key", "adjustment_set_size", "usable_error"], dropna=False)
    .size()
    .reset_index(name="rows")
)
set_summary = (
    adjustment.loc[adjustment.usable_error]
    .groupby(["adjustment_set_key", "adjustment_set_size"], dropna=False)
    .agg(
        mean_abs_error=("abs_coefficient_error", "mean"),
        median_abs_error=("abs_coefficient_error", "median"),
        rmse=("coefficient_error", lambda x: np.sqrt(np.mean(x**2))),
        signed_bias=("coefficient_error", "mean"),
        valid_rows=("coefficient_error", "size"),
        unique_edges=("edge", "nunique"),
    )
    .reset_index()
    .sort_values(
        ["mean_abs_error", "adjustment_set_key"], ascending=[False, True], kind="mergesort"
    )
)
set_quality.to_csv(OUT_DIR / "adjustment_set_data_quality.csv", index=False)
set_summary.to_csv(OUT_DIR / "adjustment_set_error_summary.csv", index=False)

membership = (
    adjustment.loc[
        adjustment.usable_error,
        ["adjustment_nodes", "coefficient_error", "abs_coefficient_error", "edge"],
    ]
    .explode("adjustment_nodes")
    .rename(columns={"adjustment_nodes": "adjustment_node"})
)
node_summary = (
    membership.dropna(subset=["adjustment_node"])
    .groupby("adjustment_node")
    .agg(
        mean_abs_error=("abs_coefficient_error", "mean"),
        median_abs_error=("abs_coefficient_error", "median"),
        rmse=("coefficient_error", lambda x: np.sqrt(np.mean(x**2))),
        signed_bias=("coefficient_error", "mean"),
        valid_rows=("coefficient_error", "size"),
        unique_edges=("edge", "nunique"),
    )
    .reset_index()
    .sort_values(["mean_abs_error", "adjustment_node"], ascending=[False, True], kind="mergesort")
)
node_summary.to_csv(OUT_DIR / "adjustment_node_error_summary.csv", index=False)

# HC3 OLS tests set-size association while controlling for the main simulation factors.
adjustment["log_n"] = np.log(adjustment["n_samples"])
adjustment["mechanism"] = adjustment["missing_data_mechanism"].astype("category")
adjustment_model = adjustment.loc[adjustment.usable_error].copy()
adjustment_formula = "abs_coefficient_error ~ adjustment_set_size + log_n + missing_data_rate + missing_edge_rate + C(mechanism)"
if HAS_STATSMODELS and len(adjustment_model) >= 10:
    adjustment_fit = smf.ols(adjustment_formula, data=adjustment_model).fit(cov_type="HC3")
    adjustment_coefficients = pd.DataFrame(
        {
            "term": adjustment_fit.params.index,
            "estimate": adjustment_fit.params.values,
            "std_error": adjustment_fit.bse.values,
            "p_value": adjustment_fit.pvalues.values,
            "ci_low": adjustment_fit.conf_int()[0].values,
            "ci_high": adjustment_fit.conf_int()[1].values,
        }
    )
    adjustment_coefficients.to_csv(OUT_DIR / "adjustment_set_model_coefficients.csv", index=False)
    pd.DataFrame(
        [
            {
                "formula": adjustment_formula,
                "n": len(adjustment_model),
                "r_squared": adjustment_fit.rsquared,
                "model_type": "OLS_HC3",
            }
        ]
    ).to_csv(OUT_DIR / "adjustment_set_model_summary.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(set_summary["adjustment_set_size"], set_summary["mean_abs_error"], alpha=0.65)
ax.set(xlabel="Adjustment-set size", ylabel="Mean absolute coefficient error")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(OUT_DIR / "adjustment_set_size_vs_error.png", dpi=150)
plt.close(fig)
display(set_summary.head(20))
display(node_summary.head(20))

,adjustment_set_key,adjustment_set_size,mean_abs_error,median_abs_error,rmse,signed_bias,valid_rows,unique_edges
328,crp|fis|hupA|ihfA|ihfB|lrp|rpoD|tyrR,8,1.161693,0.132126,10.632787,0.839547,326,1
173,arcA|fliA|fur|hfq|lrp|nac,6,0.294167,0.200093,0.407498,0.293795,326,1
164,arcA|cueR|fur|kdpE|lrp|mcbR|nac|nhaR|phoP|rcdA...,15,0.245614,0.152658,0.348952,0.243465,326,1
379,crp|gadX|lrp|nagC|rpoD|xylR,6,0.239854,0.181191,0.340775,0.228270,326,1
378,crp|fur|rpoD|zraR,4,0.238195,0.118223,0.369909,0.230325,326,1
463,dksA|fnr|nsrR|rpoD,4,0.235093,0.193390,0.293959,0.220050,326,1
400,crp|lexA|rpoD,3,0.227938,0.220101,0.293405,0.212566,326,1
635,nac|rpoD|ygiV,3,0.225110,0.174547,0.300646,0.220130,326,1
132,arcA|crp|dksA|fnr|nac|rpoD|torR|zraR,8,0.215498,0.169818,0.287677,0.185592,326,1
140,arcA|crp|fis|rpoD|rpoH|soxS,6,0.213576,0.189897,0.269090,0.193254,326,1


,adjustment_node,mean_abs_error,median_abs_error,rmse,signed_bias,valid_rows,unique_edges
154,tyrR,0.257853,0.113228,3.762437,0.205506,2608,8
150,sutR,0.206593,0.135422,0.407296,-0.158819,326,1
79,hupA,0.196541,0.092231,3.072594,0.140957,3912,12
74,hfq,0.187771,0.145833,0.265683,0.182353,1304,4
87,lacI,0.178223,0.152762,0.252780,0.176447,978,3
143,soxR,0.170335,0.154338,0.221190,0.159845,652,2
92,malT,0.167835,0.120520,0.232549,0.162332,1304,4
72,gutM,0.164957,0.154502,0.205452,0.146485,326,1
146,srlR,0.164957,0.154502,0.205452,0.146485,326,1
45,exuR,0.164914,0.107717,0.235266,-0.159551,326,1


## Interpretation and limitations

The rankings are descriptive contributors to absolute error, not causal claims. Edge totals reconcile directly to edge contributions; node incident totals intentionally count each directed edge at both endpoints. Missing or failed estimates remain visible in `error_analysis_data_quality.csv` and are never converted to zero. Adjustment-set results are associations and can be confounded by edge, graph, and condition structure; rare nodes and sets require caution. The model uses `missing_data_rate` for `r_miss`, `missing_edge_rate` for `d_edge`, and treats mechanism as categorical. `u_graph` is only fit when at least two distinct `scm_seed` values are present; otherwise the notebook reports the non-identifiable random effect and uses HC3 OLS when available. Single graph/seed designs do not support empirical graph-level uncertainty or confidence intervals over graph realizations.